# IT Service Ticket Classification

This notebook demonstrates how to build a Machine Learning model to classify IT support tickets into various categories (e.g., Hardware, Access, Software) using text data.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
nltk.download("stopwords")

## 1. Load Data

In [ ]:
# Load the IT Service Ticket dataset
df = pd.read_csv("../data/raw/all_tickets.csv")
df.head()

In [ ]:
print(f"Dataset Shape: {df.shape}")
df.info()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Rename columns for clarity
# Expected columns: 'Document' -> text, 'Topic_group' -> label
if 'Document' in df.columns and 'Topic_group' in df.columns:
    df = df.rename(columns={'Document': 'text', 'Topic_group': 'label'})

# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y="label", data=df, order=df['label'].value_counts().index)
plt.title("Distribution of Ticket Categories")
plt.show()

## 3. Data Preprocessing

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove punctuation/numbers
    text = re.sub(r"[^a-z\s]", "", text)
    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

print("Cleaning text...")
df["clean_text"] = df["text"].apply(clean_text)
df.head()

## 4. Model Training

In [ ]:
# Remove empty rows if any
df = df[df["clean_text"] != ""]

X = df["clean_text"]
y = df["label"]

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Vectorize Text (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
print("Model training complete.")

## 5. Evaluation

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix Visualization
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 6. Real-time Inference Example

In [ ]:
def assign_priority(text, category):
    """
    Assigns a priority level (High, Medium, Low) based on text keywords and category.
    """
    text = text.lower()
    
    # Critical keywords -> HIGH
    high_keywords = ["urgent", "immediately", "critical", "down", "crash", "security", "breach", "hack", "fire", "emergency", "fail", "broken"]
    
    # Keywords -> MEDIUM
    medium_keywords = ["error", "issue", "bug", "slow", "access", "login", "password", "reset", "update", "performance", "wifi"]
    
    # Check Keywords
    if any(k in text for k in high_keywords): return "High"
    if any(k in text for k in medium_keywords): return "Medium"
    
    # Check Categories
    if category in ["Hardware", "Access"]: return "High"
    if category in ["Software", "System"]: return "Medium"
    
    return "Low"

def predict_ticket(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    priority = assign_priority(text, pred)
    return {"category": pred, "priority": priority}

# Test cases
sample_tickets = [
    "My mouse is broken and I need a replacement immediately",
    "I cannot login to the VPN, access denied",
    "I need to install python on my laptop",
    "System is down, critical failure!",
    "How do I request a new monitor?"
]

print("-" * 80)
print(f"{'Ticket Descriptions':<50} | {'Category':<15} | {'Priority'}")
print("-" * 80)

for ticket in sample_tickets:
    result = predict_ticket(ticket)
    print(f"{ticket[:47]+'...':<50} | {result['category']:<15} | {result['priority']}")
print("-" * 80)